# Introduction

The notebook for calculating intrument functions.
For more details see signal_processing_pipeline.ipynb.

# Libraries


In [ ]:
from mascope_lib.file_func import load_file
import numpy as np
from scipy.signal import find_peaks, peak_widths
from mascope_lib.peak import fwhm_to_sigma
import plotly.graph_objects as go
from lmfit.models import GaussianModel, SplineModel, SkewedGaussianModel
from mascope_hardware.tofwerk import TwFitResolution
from scipy.optimize import curve_fit
from matplotlib import pyplot as plt
import mascope_api
from datetime import datetime

# Functions


In [ ]:
def fit_gaussian(x, y):
    """Fit the spectrum range with a skewed Gaussian peak-shape using lmfit

    :param x: mz scale
    :type x: array-like
    :param y: counts
    :type y: array-like
    :return: ModelResult (see lmfit doc)
    :rtype: ModelResult
    """
    # Initialize fitting parameters for the main peak
    model = SkewedGaussianModel(prefix="p_")
    params = model.make_params()
    if MS == "tof":
        # Fitting parameters for the background
        knot_xvals = np.array([-dmz, -dmz / 2, -dmz / 3, dmz / 3, dmz / 2, dmz])
        bkg = SplineModel(prefix="bkg_", xknots=knot_xvals)
        params.update(bkg.guess(y, x))
        # Total model
        model = model + bkg

    params.add("p_amplitude", value=1, max=1.1)
    params.add("p_center", value=0)
    params.add("p_sigma", value=0.01)
    params.add("p_gamma", value=0.1)

    # Perform fitting
    fit = model.fit(y, params, x=x)
    return fit


def calculate_fwhm(x, y):
    # Find the maximum count and its index
    max_index = np.argmax(y)
    max_count = y[max_index]

    # Find the half maximum count
    half_max_count = max_count / 2

    # Find the indices where the counts are closest to the half maximum
    left_index = np.argmin(np.abs(y[:max_index] - half_max_count))
    right_index = np.argmin(np.abs(y[max_index:] - half_max_count)) + max_index

    # Calculate the FWHM
    fwhm = x[right_index] - x[left_index]

    return fwhm


def calculate_center_of_mass(x, y):
    # Calculate the center of mass
    center_of_mass = np.sum(x * y) / np.sum(y)
    return center_of_mass

# Load file and find peaks


In [ ]:
# Load data file
data_path = "C:/mascope_data"
file_name = "TestOrbitrap_20231227_1010_MION2_DBrMe_DHS_1ul_50pg_Explosive_Mix_60-600mz"
f = load_file(file_name)
# Define the MS
if "tof" in file_name.lower():
    MS = "tof"
elif "orbi" in file_name.lower():
    MS = "orbi"
else:
    raise BaseException(
        """
        The instrument can not be determined from the file name.
        """
    )
# Extract averaged spectrum and mz values
spec = f.signal.mean(dim="time").values
mz = f.mz.values
# Find peaks. Tune the threshold if there are no peaks/too many peaks detected.
peak, _ = find_peaks(spec, height=np.percentile(spec, 99.5))
fwhm = peak_widths(spec, peak, rel_height=0.5)[0]
sig = np.array([fwhm_to_sigma(fw) for fw in fwhm])
# Plot result
fig = go.Figure(
    [
        go.Scatter(x=mz, y=spec, name="Raw counts"),
        go.Scatter(x=mz[peak], y=spec[peak], mode="markers", name="Detected peaks"),
    ]
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)
print(f"{len(peak)} peaks were detected.")
fig.show()

# Calculate peak shape


In [ ]:
# Expected half-width band presumably containing one peak
# Decrease dmz if you get too many peaks in one band
# Increase dmz if the peak shape is distorted/cut
dmz = 0.1  # TODO find way to automatically estimate optimal dmz value
r_squared_threshold = 0.96

peak_traces = []
peak_norm_traces = []

p_x = np.linspace(-30, 30, 601)
p_ys = []
p_mzs = []
p_fwhms = []
p_centers = []

for i, p in enumerate(peak):
    p_height = spec[p]
    # Select a narrow region (peak center +/- dmz) of the spectrum around the peak
    p_mz_center = mz[p]
    p_mz0 = p_mz_center - dmz
    p_mz1 = p_mz_center + dmz
    # Find indices
    p_i0 = np.argmin(np.abs(mz - p_mz0))  # left border
    p_i1 = np.argmin(np.abs(mz - p_mz1))  # right border
    p_i = p - p_i0  # center
    # Peak region
    p_mz = mz[p_i0:p_i1]
    p_spec = spec[p_i0:p_i1]

    if np.max(p_spec) > p_height:
        # 'p' is not the biggest peak in range, dismiss
        continue

    # Normalize peak region: mz around 0 and spec to range [0, 1]
    p_mz_norm = p_mz - p_mz_center
    p_spec_norm = p_spec / p_height
    p_spec_norm -= p_spec_norm.min()

    peak_traces.append(go.Scatter(x=p_mz_norm, y=p_spec_norm))
    # Fit peak in the region
    fit = fit_gaussian(p_mz_norm, p_spec_norm)
    p_spec_norm_fit = fit.eval_components()["p_"]

    if fit.rsquared < r_squared_threshold:
        # fitting error to large, dismiss
        continue

    # Remove junk peaks arond main one
    mask = np.where(fit.best_fit < 1e-9)
    p_spec_norm[mask] = 0

    # Get peak top location

    if MS == "tof":
        # Remove baseline if TOF
        p_spec_norm -= fit.eval_components()["bkg_"]
        p_spec_norm[np.where(p_spec_norm < 0)] = 0

    top_y = np.max(p_spec_norm_fit)
    top_x = calculate_center_of_mass(p_mz_norm, p_spec_norm_fit)

    # Refine peak position and height
    p_mz_norm -= top_x
    p_spec_norm /= top_y

    # if np.abs(p_mz_norm[np.argmin(np.abs(p_spec_norm - np.max(p_spec_norm)))]) > 5e-3:
    #     continue

    # Get and store Gaussian peak sigma and width
    try:
        p_fwhm = calculate_fwhm(p_mz_norm, p_spec_norm_fit)
    except Exception:
        continue
    p_centers.append(top_x)
    p_fwhms.append(p_fwhm)
    # p_sigma = fit.params["p_sigma"].value
    p_sigma = p_fwhm / (2 * np.sqrt(2 * np.log(2)))
    # Scale peak to width sigma=1
    p_mz_norm /= p_sigma
    # Store peak position
    p_mzs.append(p_mz_center)
    # Interpolate the normalized (both width and height) peak into predefined domain "p_x"
    p_y = np.interp(p_x, p_mz_norm, p_spec_norm, left=0, right=0)
    # Store normalized and refined peak shape and add to the figure
    p_ys.append(p_y)
    peak_norm_traces.append(go.Scatter(x=p_x, y=p_y, name=f"peak {i}"))

# Clean shifted outliers
p_centers = np.array(p_centers)
non_outlier_indx = np.where(p_centers - np.median(p_centers) < 1e-3)[0]
peak_norm_traces = [peak_norm_traces[i] for i in non_outlier_indx]
p_fwhms = [p_fwhms[i] for i in non_outlier_indx]
p_ys = [p_ys[i] for i in non_outlier_indx]
p_mzs = [p_mzs[i] for i in non_outlier_indx]

# Calculate median peak shape
p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)
# Check if p_median is empty
if p_median.all() == np.nan:
    raise Exception(
        """p_median is empty
        Probably fitting error threshold is too strict"""
    )
# Add to the figure
peak_norm_traces.append(
    go.Scatter(x=p_x, y=p_median, line={"color": "red", "width": 3}, name="median")
)
# Plot the output
fig = go.FigureWidget(
    peak_norm_traces,
    {"title": f"Median peak shape of {len(peak_norm_traces)-1} peaks"},
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)


def toggle_trace_visibility(trace, points, selector):
    """Updates the median peak shape based on shapes present in the figure"""
    if not points.point_inds:
        return
    trace.visible = "legendonly"

    selected_traces = [
        trace.y
        for trace in fig.data
        if ("peak" in trace.name and trace.visible != "legendonly")
    ]

    fig.update_layout(
        title=f"Median peak shape of {len(selected_traces)} peaks",
        height=300,
        width=800,
        margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    )
    fig.update_traces(
        {"y": np.median(np.array(selected_traces), axis=0)},
        selector=lambda trace: trace.name == "median",
    )


print("Click on distorted peaks to remove them from peak shape estimation.")
[trace.on_click(toggle_trace_visibility) for trace in fig.data]
fig

Save median peak shape


In [ ]:
# Save peak
peak_shape = {"x": np.array(p_x), "y": np.array(p_median)}
p_ys = np.array(p_ys)
# Calculate residuals for normalized peaks vs median peak shape
norm_residuals = p_ys - peak_shape["y"]
## calculate R-squared vs mz
r_sq_norm = 1 - (norm_residuals**2).sum(axis=1) / ((p_ys - p_ys.mean()) ** 2).sum(
    axis=1
)
# Plot error vs mz
err_fig = plt.figure(figsize=(7, 1.5))
plt.scatter(p_mzs, r_sq_norm)
plt.xlabel("mz")
plt.ylabel(r"R$^2$")

# Calculate resolution function


In [ ]:
# Support functions
def R_tof(mz):
    return R0 - R0 / (1 + np.exp((mz - mz0) / dmz))


def R_orb(mz):
    return R_orb_coef / np.sqrt(mz)


def get_line(x, a, b):
    x = np.array(x)
    return a * x + b


def get_polynome(x, a, b, c):
    x = np.array(x)
    return a * x**2 + b * x + c


def get_inverse_sqrt(x, a):
    x = np.array(x)
    return a / np.sqrt(x)

In [ ]:
# Get the fitted line depending on the instrument
if "tof" in file_name.lower():
    regres = linregress(p_mzs, p_fwhms)
    p_fwhms_line = get_line(p_mzs, regres.slope, regres.intercept)
if "orbi" in file_name.lower():
    coefs = np.polyfit(p_mzs, p_fwhms, 2)
    p_fwhms_line = get_polynome(p_mzs, *coefs)

# Points higher than ndev * std_dev will be later filtered out
ndev = 1
# Get residuals and standard deviation
residuals = p_fwhms - p_fwhms_line
std_dev = np.std(residuals)
is_outlier = (residuals > ndev * std_dev) | (residuals < -ndev * std_dev)

# Plot figure to see outliyers
fig, (ax_res, ax_fwhm) = plt.subplots(nrows=1, ncols=2, figsize=(12, 3))
ax_fwhm.plot(p_mzs, p_fwhms, label="True FWHM")
ax_fwhm.plot(p_mzs, p_fwhms_line, linestyle="--", color="red", label="Approximation")
ax_fwhm.fill_between(
    p_mzs,
    p_fwhms_line + ndev * std_dev,
    p_fwhms_line - ndev * std_dev,
    alpha=0.2,
    label="Non-outliers",
)
ax_fwhm.set(ylabel="FWHM", xlabel="mz")
ax_fwhm.legend()

# Remove outliers
p_fwhms_filt = np.array(p_fwhms, dtype=np.double)[~is_outlier]
mass = np.array(p_mzs, dtype=np.double)[~is_outlier]
resolution = mass / p_fwhms_filt

# the intrument is TOF
if "tof" in file_name.lower():
    # Prepare arguments for TwFitResolution
    nbrPoints = len(mass)
    # Initial guesses for fitting function
    R0 = np.array([np.max(resolution)], dtype=np.double)
    mz0 = np.array(
        [mass[(np.abs(resolution - resolution.mean())).argmin()]], dtype=np.double
    )
    dmz = np.array([np.percentile(mass, 70) - np.percentile(mass, 30)], dtype=np.double)
    # Get fitted parameters for TOF resolution function
    ret = TwFitResolution(nbrPoints, mass, resolution, R0, mz0, dmz)
    R0, mz0, dmz = R0[0], mz0[0], dmz[0]

    print(f"TOF, code {ret}, R0={R0:.3f}, m0={mz0:.3f}, dm={dmz:.3f}")

    resolution_function = R_tof
# the intrument is Orbitrap
if "orbi" in file_name.lower():
    # Fit resolution with the inverse square root
    popt, pcov = curve_fit(get_inverse_sqrt, mass, resolution)
    R_orb_coef = popt[0]
    resolution_function = R_orb

    print(f"Orbi, a={R_orb_coef:.3f}")

# Plot fit
ax_res.scatter(
    p_mzs, np.array(p_mzs) / np.array(p_fwhms), color="grey", label="Ommited pairs"
)
ax_res.scatter(mass, resolution, color="black", label="Used mass/resolution pairs")
ax_res.plot(mass, resolution_function(mass), "r", label="Fitted resolution function")
ax_res.set(
    ylabel="Resolution",
    xlabel="mz",
    title="Resolution function",
)
ax_res.legend()

# Save instrument functions


In [ ]:
# Get the current UTC date and time
datetime_utc = datetime.now()
# Convert the datetime object to a string in UTC format
datetime_utc_str = datetime_utc.isoformat()

mascope_api.create_instrument_function(
    mascope_url="http://localhost:8080/",
    instrument="KORBI2",
    datetime_utc=datetime_utc_str,
    peakshape=peak_shape,
    resolution_function=resolution_function,
)